In [1]:
import pandas as pd
import numpy as np

In [2]:
train_data = pd.read_csv("data/raw/train.csv")
test_data = pd.read_csv("data/raw/test.csv")

train_data['is_train'] = 1
test_data['is_train'] = 0

data = pd.concat([train_data, test_data], ignore_index=True, sort=False)

In [3]:
data["title"] = data["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
data["title"] = data["title"].replace(['Mlle', 'Ms'], 'Miss')
data["title"] = data["title"].replace('Mme', 'Mrs')
rare_titles = ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"]
data["title"] = data["title"].replace(rare_titles, "Rare")

In [4]:
title_age_medians = data[data['is_train'] == 1].groupby("title")["Age"].median()
global_age_median = data[data['is_train'] == 1]["Age"].median()

data["Age"] = data["Age"].fillna(data["title"].map(title_age_medians)).fillna(global_age_median)

In [5]:
pclass_fare_medians = data[data['is_train'] == 1].groupby("Pclass")["Fare"].median()
data["Fare"] = data["Fare"].fillna(data["Pclass"].map(pclass_fare_medians))

In [6]:
ticket_count = data["Ticket"].value_counts()
data["ticket_size"] = data["Ticket"].map(ticket_count)
data["family_size"] = data["SibSp"] + data["Parch"] + 1
data["group_size"] = data[["ticket_size", "family_size"]].max(axis=1)
data["is_alone"] = (data["group_size"] == 1).astype(int)
data["fare_per_person"] = data["Fare"] / data["group_size"]

In [7]:
data["deck"] = data["Cabin"].apply(lambda x: str(x)[0] if pd.notnull(x) else 'M')
data["deck"] = data["deck"].replace('T', 'M')

data["Cabin_Num"] = data["Cabin"].str.extract(r'(\d+)').astype(float)
data["ship_side"] = data["Cabin_Num"].apply(
    lambda x: 'M' if np.isnan(x) else ("Port" if int(x) % 2 == 0 else "Starboard")
)

In [8]:
embarked_mode = data[data['is_train'] == 1]["Embarked"].mode()[0]
data["Embarked"] = data["Embarked"].fillna(embarked_mode)

In [9]:
data = pd.get_dummies(data, columns=["Sex", "title", "deck", "ship_side", "Embarked"], drop_first=True)
drop_cols = ["Cabin", "Cabin_Num", "Name", "Ticket", "ticket_size", "family_size", "SibSp", "Parch"]
data = data.drop(columns=drop_cols)

In [10]:
train_df = data[data["is_train"] == 1].drop(columns=["is_train"])
test_df = data[data["is_train"] == 0].drop(columns=["is_train", "Survived"])

train_df.to_csv("data/processed/train.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)